In [1]:
import trellis2.models as t2models
from trellis2.modules.sparse import SparseTensor
from trellis2.modules.image_feature_extractor import DinoV3FeatureExtractor
from trellis2.pipelines.rembg import BiRefNet
from trellis2.pipelines.samplers import FlowEulerSampler, FlowEulerGuidanceIntervalSampler

from PIL import Image
from safetensors.torch import load_file
import json
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import math

import pyvista as pv
pv.set_jupyter_backend("html")

[SPARSE] Conv backend: flex_gemm; Attention backend: flash_attn


In [2]:
# Image preprocessing for conditiong

def preprocess_image(input: Image.Image, rembg_model) -> Image.Image:
    """
    Preprocess the input image.
    """
    # if has alpha channel, use it directly; otherwise, remove background
    has_alpha = False
    if input.mode == 'RGBA':
        alpha = np.array(input)[:, :, 3]
        if not np.all(alpha == 255):
            has_alpha = True
    max_size = max(input.size)
    scale = min(1, 1024 / max_size)
    if scale < 1:
        input = input.resize((int(input.width * scale), int(input.height * scale)), Image.Resampling.LANCZOS)
    if has_alpha:
        output = input
    else:
        input = input.convert('RGB')
        output = rembg_model(input)
    output_np = np.array(output)
    alpha = output_np[:, :, 3]
    bbox = np.argwhere(alpha > 0.8 * 255)
    bbox = np.min(bbox[:, 1]), np.min(bbox[:, 0]), np.max(bbox[:, 1]), np.max(bbox[:, 0])
    center = (bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2
    size = max(bbox[2] - bbox[0], bbox[3] - bbox[1])
    size = int(size * 1)
    bbox = center[0] - size // 2, center[1] - size // 2, center[0] + size // 2, center[1] + size // 2
    output = output.crop(bbox)  # type: ignore
    output = np.array(output).astype(np.float32) / 255
    output = output[:, :, :3] * output[:, :, 3:4]
    output = Image.fromarray((output * 255).astype(np.uint8))
    return output

In [3]:
ckpt_dir = Path("/flux/vault/pretrained_checkpoints/trellis")

def load_model(model_ckpt: str):
    with open(ckpt_dir / (model_ckpt + ".json")) as f:
        model_args = json.load(f)
    model = getattr(t2models, model_args["name"])(**model_args['args'])
    state_dict = load_file(ckpt_dir / (model_ckpt + ".safetensors"))
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    return model

In [4]:
# Sparse/voxel latent generator and decoder
ss_flow_model = load_model("ss_flow_img_dit_1_3B_64_bf16")
ss_decoder = load_model("ss_dec_conv3d_16l8_fp16")

# Structured latent sampler and decoder
RESOLUTION = 512 # Resolution of slats, 512 or 1024
slat_flow_model = load_model(f"slat_flow_img2shape_dit_1_3B_{RESOLUTION}_bf16")
slat_decoder = load_model("shape_dec_next_dc_f16c32_fp16")
slat_decoder.set_resolution(RESOLUTION)

[ATTENTION] Using backend: flash_attn


In [22]:
NUM_SAMPLES = 1
SAMPLE_STEPS = 12

CONDITIONAL = False

cond = torch.zeros((NUM_SAMPLES, 1, 1024)).cuda()
if CONDITIONAL:
    # Optionally include image features
    image = Image.open("assets/example_image/T.png").convert("RGB")
    # will need to request access to the dinov3 HF repo
    dinov3 = DinoV3FeatureExtractor(Path("/flux/vault/pretrained_checkpoints/dino"))
    rembg_model = BiRefNet()
    rembg_model.cuda()
    image = preprocess_image(image, rembg_model)
    plt.imshow(image)
    rembg_model.cpu()
    dinov3.cuda()
    cond = dinov3([image]).repeat(NUM_SAMPLES, 1, 1)
    dinov3.cpu()

sampler = FlowEulerGuidanceIntervalSampler(sigma_min=1e-5)

In [33]:
# Sample sparse structure latents: ie a latent representation of the voxel grid
ss_flow_model.cuda()

reso = ss_flow_model.resolution  # 16 (512) or 32 (1024)
in_channels = ss_flow_model.in_channels
noise = torch.randn(NUM_SAMPLES, in_channels, reso, reso, reso).cuda()

ss_samples = sampler.sample(
    ss_flow_model,
    noise,
    cond=cond,
    neg_cond=torch.zeros_like(cond),
    steps=SAMPLE_STEPS,
    guidance_strength=7.5,
    guidance_rescale=0.7,
    guidance_interval=[0.6, 1.0],
    rescale_t=5.0,
).samples

ss_flow_model.cpu()
torch.cuda.empty_cache()

print(ss_samples.shape)  # (NUM_SAMPLES, in_channels, reso, reso, reso)
# print the ranges of the samples
print("Sparse structure latent ranges:", ss_samples.min().item(), ss_samples.max().item())

Sampling: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:02<00:00,  5.33it/s]


torch.Size([1, 8, 16, 16, 16])
Sparse structure latent ranges: -3.7423248291015625 2.523146629333496


In [24]:
# Decode sparse structure latents to full voxel grid
ss_decoder.cuda()

decoded = ss_decoder(ss_samples) > 0  # (NUM_SAMPLES, 1, decoder_reso, decoder_reso, decoder_reso), decoder_reso=64
if slat_flow_model.resolution != decoded.shape[2]:
    ratio = decoded.shape[2] // slat_flow_model.resolution
    decoded = torch.nn.functional.max_pool3d(decoded.float(), ratio, ratio, 0) > 0.5
coords = torch.argwhere(decoded)[:, [0, 2, 3, 4]].int()
ss_decoder.cpu()
torch.cuda.empty_cache()
print(coords.shape)  # (num_points, 4), first column is batch index

torch.Size([1727, 4])


In [32]:
# Plot first sample
coords_np = coords.cpu().numpy()
mask = coords_np[:, 0] == 1
xyz = coords_np[mask, 1:4].astype(float)

cloud = pv.PolyData(xyz)
cube = pv.Cube(x_length=1, y_length=1, z_length=1)
voxels = cloud.glyph(geom=cube, scale=False, orient=False)
voxels.plot(show_edges=True)

ValueError: Empty meshes cannot be plotted. Input mesh has zero points. To allow plotting empty meshes, set `pv.global_theme.allow_empty_mesh = True`

In [27]:
# Sample latents on each occupied voxel (ie structured latents (ie slats))
slat_flow_model.cuda()

noise = SparseTensor(
            feats=torch.randn(coords.shape[0], slat_flow_model.in_channels).cuda(),
            coords=coords,
        )

slats = sampler.sample(
    model=slat_flow_model,
    noise=noise,
    cond=cond,
    neg_cond=torch.zeros_like(cond),
    steps=SAMPLE_STEPS,
    guidance_strength=7.5,
    guidance_rescale=0.7,
    guidance_interval=[0.6, 1.0],
    rescale_t=5.0,
).samples

slat_flow_model.cpu()
torch.cuda.empty_cache()

def normalize_slats(slats):
    mean = [
            0.781296, 0.018091, -0.495192, -0.558457, 1.060530, 0.093252, 1.518149, -0.933218,
            -0.732996, 2.604095, -0.118341, -2.143904, 0.495076, -2.179512, -2.130751, -0.996944,
            0.261421, -2.217463, 1.260067, -0.150213, 3.790713, 1.481266, -1.046058, -1.523667,
            -0.059621, 2.220780, 1.621212, 0.877230, 0.567247, -3.175944, -3.186688, 1.578665
        ]
    std = [
            5.972266, 4.706852, 5.445010, 5.209927, 5.320220, 4.547237, 5.020802, 5.444004,
            5.226681, 5.683095, 4.831436, 5.286469, 5.652043, 5.367606, 5.525084, 4.730578,
            4.805265, 5.124013, 5.530808, 5.619001, 5.103930, 5.417670, 5.269677, 5.547194,
            5.634698, 5.235274, 6.110351, 5.511298, 6.237273, 4.879207, 5.347008, 5.405691
        ]
    
    std = torch.tensor(std)[None].cuda()
    mean = torch.tensor(mean)[None].cuda()
    
    return slats * std + mean

slats = normalize_slats(slats)

print(slats.shape)  # (NUM_SAMPLES, 32) this is just how the SparseTensor object is defined

Sampling: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:01<00:00,  6.07it/s]


torch.Size([1, 32])


In [28]:
# Decode slats to mesh
slat_decoder.cuda()
decoded_slats = slat_decoder(slats, return_subs=False)
slat_decoder.cpu()
torch.cuda.empty_cache()
print(decoded_slats)  # List[Mesh] ! via flexicubes

In [29]:
decoded_slats

In [30]:
TARGET_VERTS = 100_000

n = len(decoded_slats)
cols = math.ceil(math.sqrt(n))
rows = math.ceil(n / cols)

plotter = pv.Plotter(shape=(rows, cols))

for i, mesh in enumerate(decoded_slats):
    r, c = divmod(i, cols)
    plotter.subplot(r, c)

    mesh.fill_holes()
    verts = mesh.vertices.detach().cpu().numpy()
    faces = mesh.faces.detach().cpu().numpy()
    pv_mesh = pv.PolyData.from_regular_faces(verts, faces)

    if verts.shape[0] > TARGET_VERTS:
        reduction = 1.0 - (TARGET_VERTS / verts.shape[0])
        pv_mesh = pv_mesh.decimate_pro(reduction)

    plotter.add_mesh(pv_mesh, show_edges=False)

plotter.link_views()
plotter.show()

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…